Welcome to the final week! I have cleaned our data and found great insights.
Now, it is time to build a Machine Learning model. Since our main goal is to
predict if a shipment will be late, we are solving a &#39;Classification&#39; problem
(predicting a Yes/No outcome for the &#39;delayed&#39; column). We will use a Random
Forest model, check how accurate it is, and then write down business strategies
based on what the model tells us.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
#Loading the cleaned dataset
df = pd.read_csv('Delivery_Logistics_Cleaned.csv')

### Step 1: Prepare the Data for Machine Learning
Machine learning models only understand numbers. They cannot read words like
&#39;rainy&#39; or &#39;scooter&#39;. 
To fix this, we use a trick called One-Hot Encoding.
This simply turns our text categories into columns of 0s and 1s. 
For example, if the weather is rainy, the &#39;is_rainy&#39; column gets a 1, and the &#39;is_clear&#39; column
gets a 0.

In [ ]:
#Data Prep: Convert text into numbers (One-Hot Encoding)
# We drop columns we don't need for predicting like delivery_id and expected_time
features = ['delivery_partner', 'vehicle_type', 'weather_condition', 'region',
            'distance_km', 'package_weight_kg']
X = pd.get_dummies(df[features], drop_first=True)


# Our target is the 'delayed' column. Let's convert 'yes'/'no' to 1 and 0.
y = df['delayed'].apply(lambda x: 1 if x == 'yes' else 0)

In [4]:
X.head()

,distance_km,package_weight_kg,delivery_partner_blue dart,delivery_partner_delhivery,delivery_partner_dhl,delivery_partner_ecom express,delivery_partner_ekart,delivery_partner_fedex,delivery_partner_shadowfax,delivery_partner_xpressbees,...,vehicle_type_van,weather_condition_cold,weather_condition_foggy,weather_condition_hot,weather_condition_rainy,weather_condition_stormy,region_east,region_north,region_south,region_west
0,297.0,46.96,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
1,89.6,47.39,False,False,False,False,False,False,False,True,...,False,True,False,False,False,False,False,False,False,False
2,273.5,26.89,False,False,False,False,False,False,True,False,...,False,False,False,False,True,False,True,False,False,False
3,269.7,12.69,False,False,True,False,False,False,False,False,...,False,True,False,False,False,False,True,False,False,False
4,256.7,37.02,False,False,True,False,False,False,False,False,...,True,False,True,False,False,False,False,True,False,False


In [5]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: delayed, dtype: int64

### Step 2: Split the Data into Train and Test Sets
If we show the model all our data to learn, we won&#39;t have any hidden data left
to test if it actually got smart or just memorized the answers. So, we split
the dataset: 80% is used to &#39;train&#39; or teach the model, and 20% is held back to
&#39;test&#39; it.

In [ ]:
#Split the data (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

In [7]:
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")
print(f"Number of features: {X_train.shape[1]}")


Training set size: 20000 samples
Testing set size: 5000 samples
Number of features: 24


### Step 3: Train the Random Forest Model
I am using a Random Forest Classifier. Instead of relying on just one
decision path, this model creates a &#39;forest&#39; of many different decision trees.
When a new delivery comes in, all the trees look at the distance, weight, and
weather, and they &#39;vote&#39; on whether it will be delayed. It is highly accurate
and perfect for supply chain problems.

In [8]:
#Train the Random Forest Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [9]:
#Make predictions on the test data
y_pred = model.predict(X_test)

### Step 4: Evaluate Model Performance
I need to know how good our model is. We will look at &#39;Accuracy&#39; (what
percentage of packages did it guess correctly?) and we will print a
&#39;Classification Report&#39; to see how well it catches the actual delays.

In [13]:
#Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")

Model Accuracy: 71.48%



In [14]:
print("Detailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['On Time (0)',
                                                              'Delayed (1)']))

Detailed Classification Report:
              precision    recall  f1-score   support

 On Time (0)       0.76      0.90      0.82      3677
 Delayed (1)       0.42      0.20      0.27      1323

    accuracy                           0.71      5000
   macro avg       0.59      0.55      0.55      5000
weighted avg       0.67      0.71      0.68      5000



In [15]:
#Check which features are the most important for causing delays
importances = pd.DataFrame({'Feature': X.columns, 'Importance': model.feature_importances_})
importances = importances.sort_values(by='Importance', ascending=False).head(5)
print("\nTop 5 Drivers of Delivery Delays:")
print(importances)


Top 5 Drivers of Delivery Delays:
                     Feature  Importance
0                distance_km    0.348357
1          package_weight_kg    0.320205
19  weather_condition_stormy    0.025726
22              region_south    0.020090
20               region_east    0.018938


### The Random Forest model achieved an overall accuracy of 71.48%. A deeper look at the classification report reveals that while the model is highly effective at identifying on-time shipments (90% recall), it is conservative when predicting delays, catching only 20% of actual late deliveries. This indicates a class imbalance that could be improved in future iterations using techniques like SMOTE or adjusting class weights.

### Feature importance scores highlight that physical logistics constraints—specifically delivery distance (0.348) and package weight (0.320)—drive over 66% of the model's decision-making. Among external variables, stormy weather (0.025) emerged as the single highest categorical risk factor.

### Based on these findings, the business should implement strict weight-to-distance dispatch thresholds to prevent overburdening vehicles on long routes. Additionally, deploying dynamic weather routing—such as automatically pausing scooter dispatches and switching to enclosed vans when storms are forecasted—will directly mitigate the primary environmental cause of network delays.